# Scenario: Spotting "The Great Shift"

In [4]:
import pandas as pd
import sqlite3
# creating the dataset showing wait times before and after a system switch
wait_time_data = {
    "log_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "log_date": ["2026-05-01", "2026-05-03", "2026-05-05", "2026-05-08", 
                "2026-05-12", "2026-05-13", "2026-05-15", "2026-05-16"],
    "recorded_wait_minutes": [15, 12, 14, 13, 45, 50, 48, 52], # notice the jump in waiting period
    "system_version": ["Legacy", "Legacy", "Legacy", "Legacy", "New-EHR", "New-EHR", "New-EHR", "New-EHR"]
}
# adding the dataset to the DataFrame
df_wait_time = pd.DataFrame(wait_time_data)
# creating sql to save dataframe on the temporary memory
connt = sqlite3.connect(":memory:")
df_wait_time.to_sql("waits", connt, index = False, if_exists = "replace")
# creating a function to run query
def run_query(query):
    return pd.read_sql_query(query, connt)
print("********************************** Day 15 Data Drift Database is ready *******************")

********************************** Day 15 Data Drift Database is ready *******************


# The "Before vs. After" Summary

In [8]:
# query for all datase to review
all_data = "SELECT * FROM waits"
print("***************************************** all data to review *******************")
display(run_query(all_data))
print()
# query that calculates the AVERAGE recorded_wait_minutes grouped by the system_version.
average_recorded_wait_times = """
SELECT system_version, AVG(recorded_wait_minutes)  FROM waits
GROUP BY system_version
"""
print("*********************************** AVERAGE recorded_wait_minutes ***********************")
display(run_query(average_recorded_wait_times))

***************************************** all data to review *******************


,log_id,log_date,recorded_wait_minutes,system_version
0,1,2026-05-01,15,Legacy
1,2,2026-05-03,12,Legacy
2,3,2026-05-05,14,Legacy
3,4,2026-05-08,13,Legacy
4,5,2026-05-12,45,New-EHR
5,6,2026-05-13,50,New-EHR
6,7,2026-05-15,48,New-EHR
7,8,2026-05-16,52,New-EHR



*********************************** AVERAGE recorded_wait_minutes ***********************


,system_version,AVG(recorded_wait_minutes)
0,Legacy,13.50
1,New-EHR,48.75


# Identifying the "Drift Date"

In [14]:
# query to find the exact log_date where the wait time first exceeded 30 minutes to get the earliest date
drift_data_1 = """
SELECT MIN(log_date) AS first_drift_date, recorded_wait_minutes, system_version 
FROM waits
WHERE recorded_wait_minutes > 30
GROUP BY log_date, recorded_wait_minutes, system_version
"""
# query to find the exact log_date where the wait time first exceeded 30 minutes to get the first row
drift_data_2 = """
SELECT log_date, recorded_wait_minutes, system_version
FROM waits
WHERE recorded_wait_minutes > 30
ORDER BY log_date ASC
LIMIT 1
"""
print("************************** the drift date as the earliest date *****************")
display(run_query(drift_data_1))
print()
print("************************** the drift date as the single first row *****************")
display(run_query(drift_data_2))


************************** the drift date as the earliest date *****************


,first_drift_date,recorded_wait_minutes,system_version
0,2026-05-12,45,New-EHR
1,2026-05-13,50,New-EHR
2,2026-05-15,48,New-EHR
3,2026-05-16,52,New-EHR



************************** the drift date as the single first row *****************


,log_date,recorded_wait_minutes,system_version
0,2026-05-12,45,New-EHR
